# Homework 7 | CFRM 505 | Revtsov

In [15]:
import numpy as np
import scipy.stats as sps

np.random.seed(42)

# Problem 1

Suppose that $X\sim\textrm{Exp}(1/2)$ and $Y\sim\textrm{Exp}(1/3)$ are independent random variables.  Consider the probability

$$\mathbb{P}\left[X + Y > 5\right].$$

For each part, implement your method using at least $100,000$ samples.

## Part a

Estimate this probability directly by simulating both $X$ and $Y$.  Report your estimate along with the standard error and a 95\% confidence interval.

## Part b

Estimate this probability using conditional Monte Carlo.  Choose whether to condition on $X$ or $Y$ **before** running any simulations and justify your choice.  Report your estimate along with the standard error and a 95\% confidence interval.  How much did this reduce the error from part (b)?

### Solution

#### Part a

Direct Monte Carlo: simulate both $X$ and $Y$ and evaluate the indicator $\mathbf{1}[X + Y > 5]$.

In [16]:
N1 = 100_000

U1_X = np.random.uniform(0, 1, size=N1)
U1_Y = np.random.uniform(0, 1, size=N1)
X1 = -2 * np.log(U1_X)  # Exp(1/2)
Y1 = -3 * np.log(U1_Y)  # Exp(1/3)

f1_a = (X1 + Y1 > 5).astype(float)
theta1_a = np.mean(f1_a)
se1_a = np.std(f1_a, ddof=1) / np.sqrt(N1)
ci1_a = (theta1_a - 1.96 * se1_a, theta1_a + 1.96 * se1_a)

print(f"Estimate:           {theta1_a:.6f}")
print(f"Standard error:     {se1_a:.6f}")
print(f"95% CI:             ({ci1_a[0]:.6f}, {ci1_a[1]:.6f})")

Estimate:           0.402200
Standard error:     0.001551
95% CI:             (0.399161, 0.405239)


#### Part b

We condition on $X$ rather than $Y$. By the rule of thumb from the notes (Section 16.1), we should condition on the lower-variance variable: $\text{Var}[X] = 1/(1/2)^2 = 4$ versus $\text{Var}[Y] = 1/(1/3)^2 = 9$. The lower-variance variable contributes less randomness to the estimator $g(X) = \mathbb{E}[\mathbf{1}[X+Y>5] \mid X]$, leaving us with a lower-variance estimator than conditioning on $Y$ would.

Given $X = x$, we need $Y > 5 - x$. Since $Y \sim \text{Exp}(1/3)$, its survival function is $\mathbb{P}[Y > t] = e^{-t/3}$ for $t \geq 0$ and $1$ for $t < 0$. Therefore:

$$\mathbb{P}[X + Y > 5 \mid X = x] = \mathbb{P}[Y > 5 - x] = \begin{cases} e^{-(5-x)/3} & x < 5 \\ 1 & x \geq 5 \end{cases} = \min\!\left(e^{-(5-x)/3},\, 1\right)$$

We simulate only $X$ and replace the 0/1 indicator with this closed-form conditional probability.

In [17]:
U1_b = np.random.uniform(0, 1, size=N1)
X1_b = -2 * np.log(U1_b)  # Exp(1/2)
f1_b = np.minimum(np.exp(-(5 - X1_b) / 3), 1)

theta1_b = np.mean(f1_b)
se1_b = np.std(f1_b, ddof=1) / np.sqrt(N1)
ci1_b = (theta1_b - 1.96 * se1_b, theta1_b + 1.96 * se1_b)
pct1_b = (se1_b - se1_a) / se1_a * 100

print(f"Estimate:           {theta1_b:.6f}")
print(f"Standard error:     {se1_b:.6f}")
print(f"95% CI:             ({ci1_b[0]:.6f}, {ci1_b[1]:.6f})")
print(f"SE reduction:       {pct1_b:.1f}% vs part (a)")

Estimate:           0.402348
Standard error:     0.000772
95% CI:             (0.400835, 0.403861)
SE reduction:       -50.2% vs part (a)


# Problem 2

Suppose $X\sim U(-1, 1)$ and $Y\sim U(-1, 1)$ are i.i.d. random variables.  Consider the probability

$$\mathbb{P}\left[X^2 + 2Y^2 < 2\right].$$

For each part, implement y our method using at least $100,000$ samples.

## Part a

Estimate this probability directly by simulating both $X$ and $Y$.  Report your estimate along with the standard error and a 95\% confidence interval.

## Part b

Estimate this probability using conditional Monte Carlo by conditioning on $X$.  Report your estimate along with the standard error and a 95\% confidence interval.  How much did this reduce the error from part (a)?

## Part c

Estimate this probability using conditional Monte Carlo by conditioning on $Y$.  Report your estimate along with the standard error and a 95\% confidence interval.  How much did this reduce the error from part (a)?

### Solution

#### Part a

Direct Monte Carlo: simulate both $X$ and $Y$ and evaluate the indicator $\mathbf{1}[X^2 + 2Y^2 < 2]$.

In [18]:
N2 = 100_000

X2_a = np.random.uniform(-1, 1, size=N2)
Y2_a = np.random.uniform(-1, 1, size=N2)

f2_a = (X2_a**2 + 2*Y2_a**2 < 2).astype(float)
theta2_a = np.mean(f2_a)
se2_a = np.std(f2_a, ddof=1) / np.sqrt(N2)
ci2_a = (theta2_a - 1.96*se2_a, theta2_a + 1.96*se2_a)

print(f"Estimate:           {theta2_a:.6f}")
print(f"Standard error:     {se2_a:.6f}")
print(f"95% CI:             ({ci2_a[0]:.6f}, {ci2_a[1]:.6f})")

Estimate:           0.909080
Standard error:     0.000909
95% CI:             (0.907298, 0.910862)


#### Part b

Condition on $X = x$. Given $X = x$, we need $2Y^2 < 2 - x^2$, which rearranges as:

$$2Y^2 < 2 - x^2 \;\Longrightarrow\; Y^2 < \frac{2 - x^2}{2} \;\Longrightarrow\; |Y| < \sqrt{\tfrac{2-x^2}{2}}$$

For $Y \sim U(-1, 1)$, $\mathbb{P}[|Y| < t] = \mathbb{P}[-t < Y < t] = 2t/2 = t$ for $t \in [0, 1]$. Setting $t = \sqrt{(2-x^2)/2}$, we verify $t \leq 1$: since $x^2 \geq 0$, we have $(2-x^2)/2 \leq 1$, so $t \leq 1$. Therefore:

$$\mathbb{P}\!\left[X^2 + 2Y^2 < 2 \mid X = x\right] = \sqrt{\tfrac{2-x^2}{2}}$$

As $x$ ranges over $(-1, 1)$, this lies in $[1/\sqrt{2},\, 1]$: it equals 1 at $x = 0$ and $1/\sqrt{2}$ at $|x| = 1$. The conditional probability is always strictly between 0 and 1 and has lower variance than the indicator.

In [19]:
X2_b = np.random.uniform(-1, 1, size=N2)
f2_b = np.sqrt((2 - X2_b**2) / 2)

theta2_b = np.mean(f2_b)
se2_b = np.std(f2_b, ddof=1) / np.sqrt(N2)
ci2_b = (theta2_b - 1.96*se2_b, theta2_b + 1.96*se2_b)
pct2_b = (se2_b - se2_a) / se2_a * 100

print(f"Estimate:           {theta2_b:.6f}")
print(f"Standard error:     {se2_b:.6f}")
print(f"95% CI:             ({ci2_b[0]:.6f}, {ci2_b[1]:.6f})")
print(f"SE reduction:       {pct2_b:.1f}% vs part (a)")

Estimate:           0.909084
Standard error:     0.000269
95% CI:             (0.908557, 0.909611)
SE reduction:       -70.4% vs part (a)


#### Part c

Condition on $Y = y$. Given $Y = y$, we need $X^2 < 2 - 2y^2$, i.e., $|X| < \sqrt{2-2y^2}$. For $X \sim U(-1, 1)$, $\mathbb{P}[|X| < t] = \min(t, 1)$, so:

$$\mathbb{P}\!\left[X^2 + 2Y^2 < 2 \mid Y = y\right] = \min\!\left(\sqrt{2-2y^2},\;1\right)$$

The min equals 1 when $\sqrt{2-2y^2} \geq 1$, i.e., $2 - 2y^2 \geq 1$, i.e., $|y| \leq 1/\sqrt{2}$. For $|y| > 1/\sqrt{2}$ the conditional probability falls from 1 to 0 as $|y| \to 1$. This sharp transition makes it more variable than the smooth conditional probability in part (b), giving a smaller variance reduction.

In [20]:
Y2_c = np.random.uniform(-1, 1, size=N2)
f2_c = np.minimum(np.sqrt(2 - 2*Y2_c**2), 1)

theta2_c = np.mean(f2_c)
se2_c = np.std(f2_c, ddof=1) / np.sqrt(N2)
ci2_c = (theta2_c - 1.96*se2_c, theta2_c + 1.96*se2_c)
pct2_c = (se2_c - se2_a) / se2_a * 100

print(f"Estimate:           {theta2_c:.6f}")
print(f"Standard error:     {se2_c:.6f}")
print(f"95% CI:             ({ci2_c[0]:.6f}, {ci2_c[1]:.6f})")
print(f"SE reduction:       {pct2_c:.1f}% vs part (a)")

Estimate:           0.908810
Standard error:     0.000601
95% CI:             (0.907633, 0.909988)
SE reduction:       -33.9% vs part (a)


# Problem 3

Consider the quantity $\theta = \mathbb{E}[X]$ where $X\sim\textrm{Exp}(\lambda)$.  In class, we simulated $X$ with the inverse-transform method by setting

$$X = -\frac{\ln U}{\lambda},$$

where $U\sim U(0, 1)$ and we used $U$ as a control variate.  Repeat this method, but now use both $U$ and $U^2$ as control variates (i.e., two controls at once, not two separate methods each with a different control).  You need to derive the optimal values of $c$, but you can use any results from the notes without proof.  Use $\lambda = 1$.  Report your estimate along with the standard error and a 95\% confidence interval.

### Solution

We use $X = -\ln U$ with $U \sim U(0, 1)$ and two controls $Z_1 = U$, $Z_2 = U^2$. The corrected estimator is

$$\hat\theta_c = X + c_1\!\left(U - \tfrac{1}{2}\right) + c_2\!\left(U^2 - \tfrac{1}{3}\right)$$

with optimal coefficients $\mathbf{c}^* = -\Sigma_{ZZ}^{-1}\,\Sigma_{XZ}$

**Computing $\Sigma_{ZZ}$.**  Using $E[U^k] = 1/(k+1)$:

$$\text{Var}[U] = \tfrac{1}{3} - \tfrac{1}{4} = \tfrac{1}{12}, \qquad \text{Var}[U^2] = \tfrac{1}{5} - \tfrac{1}{9} = \tfrac{4}{45}, \qquad \text{Cov}[U, U^2] = \tfrac{1}{4} - \tfrac{1}{2}\cdot\tfrac{1}{3} = \tfrac{1}{12}$$

**Computing $\Sigma_{XZ}$.**  We need $E[X U^k] = E[-U^k \ln U] = \int_0^1 -u^k \ln u\, du$. Integrating by parts:

$$\int_0^1 -u^k \ln u\, du = \left[-\frac{u^{k+1}}{k+1}\ln u\right]_0^1 + \int_0^1 \frac{u^k}{k+1}\,du = 0 + \frac{1}{(k+1)^2}$$

So $E[XU] = 1/4$ and $E[XU^2] = 1/9$. Since $E[X] = 1$, $E[U] = 1/2$, $E[U^2] = 1/3$:

$$\text{Cov}[X, U] = \tfrac{1}{4} - 1\cdot\tfrac{1}{2} = -\tfrac{1}{4}, \qquad \text{Cov}[X, U^2] = \tfrac{1}{9} - 1\cdot\tfrac{1}{3} = -\tfrac{2}{9}$$

**Solving for $\mathbf{c}^*$.**  The system $\Sigma_{ZZ}\,\mathbf{c}^* = -\Sigma_{XZ}$ is:

$$\begin{bmatrix} \tfrac{1}{12} & \tfrac{1}{12} \\[3pt] \tfrac{1}{12} & \tfrac{4}{45} \end{bmatrix} \begin{bmatrix} c_1 \\ c_2 \end{bmatrix} = \begin{bmatrix} \tfrac{1}{4} \\[3pt] \tfrac{2}{9} \end{bmatrix}$$

$$\frac{3 - c_2}{12} + \frac{4c_2}{45} = \frac{2}{9} \;\Longrightarrow\; c_2\!\left(\frac{4}{45} - \frac{1}{12}\right) = \frac{2}{9} - \frac{1}{4} \;\Longrightarrow\; c_2\cdot\frac{1}{180} = -\frac{1}{36} \;\Longrightarrow\; c_2 = -5, \quad c_1 = 8$$

In [21]:
N3 = 100_000

U3 = np.random.uniform(0, 1, size=N3)
X3 = -np.log(U3)  # Exp(1) via inverse transform

Sigma_ZZ = np.array([[1/12, 1/12],
                      [1/12, 4/45]])
Sigma_XZ = np.array([-1/4, -2/9])
c_star = -np.linalg.solve(Sigma_ZZ, Sigma_XZ)

f3 = X3 + c_star[0] * (U3 - 0.5) + c_star[1] * (U3**2 - 1/3)

theta3 = np.mean(f3)
se3 = np.std(f3, ddof=1) / np.sqrt(N3)
ci3 = (theta3 - 1.96*se3, theta3 + 1.96*se3)

print(f"Optimal c:          [{c_star[0]:.4f}, {c_star[1]:.4f}]")
print(f"Estimate:           {theta3:.6f}")
print(f"Standard error:     {se3:.6f}")
print(f"95% CI:             ({ci3[0]:.6f}, {ci3[1]:.6f})")

Optimal c:          [8.0000, -5.0000]
Estimate:           0.999174
Standard error:     0.001049
95% CI:             (0.997119, 1.001229)


# Problem 4

Consider the probability

$$\theta = \mathbb{P}\left[Z > 5\right],$$

where $Z \sim N(0, 1)$.  We calculated this in class, both with the direct method (which worked very poorly) and with importance sampling by sampling from $Y\sim N(\mu, 1)$ instead of $X$.  In this problem, you will use importance sampling to calculate $\theta$ by sampling from $Y\sim N(0, \sigma^2)$.

## Part a

Find the likelihood ratio $f_{X}(Y)/f_{Y}(Y)$ for arbitrary $\sigma$.

## Part b

Implement your method to estimate $\theta$ using $\sigma = 5$.  Use a sample size of at least $100,000$.  Report your estimate along with the standard error and a 95\% confidence interval.

### Solution

#### Part a

With $X \sim N(0,1)$ and proposal $Y \sim N(0, \sigma^2)$:

$$\frac{f_X(y)}{f_Y(y)} = \frac{\frac{1}{\sqrt{2\pi}} e^{-y^2/2}}{\frac{1}{\sigma\sqrt{2\pi}} e^{-y^2/(2\sigma^2)}} = \sigma\exp\!\left(\frac{y^2}{2}\!\left(\frac{1}{\sigma^2}-1\right)\right)$$

#### Part b

With $\sigma = 5$, draw $Y_i \sim N(0, 25)$ and estimate:

$$\hat\theta = \frac{1}{N}\sum_{i=1}^{N} \mathbf{1}[Y_i > 5]\cdot 5\exp\!\left(\frac{Y_i^2}{2}\!\left(\frac{1}{25} - 1\right)\right)$$

In [22]:
N4 = 100_000

sigma = 5
Y4 = np.random.normal(0, sigma, size=N4)
lr4 = sigma * np.exp(Y4**2 / 2 * (1/sigma**2 - 1))
f4 = (Y4 > 5) * lr4

theta4 = np.mean(f4)
se4 = np.std(f4, ddof=1) / np.sqrt(N4)
ci4 = (theta4 - 1.96*se4, theta4 + 1.96*se4)

print(f"Estimate:           {theta4:.4e}")
print(f"Standard error:     {se4:.4e}")
print(f"95% CI:             ({ci4[0]:.4e}, {ci4[1]:.4e})")
print(f"True value:         {sps.norm.sf(5):.4e}")

Estimate:           2.8702e-07
Standard error:     6.6796e-09
95% CI:             (2.7393e-07, 3.0011e-07)
True value:         2.8665e-07


# Problem 5

Consider the probability

$$\theta = \mathbb{P}\left[X > 20\right]$$

where $X\sim\textrm{Exp}(1)$.

## Part a

Estimate $\theta$ using the direct method with a sample size of $N = 1,000,000$.  Report your estimate along with the standard error and a 95\% confidence interval.

## Part b

Estimate $\theta$ using importance sampling by sampling from $Y\sim\textrm{Exp}(\lambda)$ with $\lambda = 1/10$ instead of $X$ (and adjusting by the appropriate likelihood ratio).  Report your estimate along with the standard error and a 95\% confidence interval.

## Part c

Repeat part (b) using $\lambda = 1/20$ and $\lambda = 1/30$.  Which of the three choices of $\lambda$ ($1/10$, $1/15$ or $1/20$) results in the smallest standard error?

### Solution

#### Part a

Direct Monte Carlo with $N = 1{,}000{,}000$. The true value is $e^{-20} \approx 2.06 \times 10^{-9}$.

In [23]:
N5_a  = 1_000_000
N5_is = 100_000

U5 = np.random.uniform(0, 1, size=N5_a)
X5 = -np.log(U5)  # Exp(1)
f5_a = (X5 > 20).astype(float)

theta5_a = np.mean(f5_a)
se5_a = np.std(f5_a, ddof=1) / np.sqrt(N5_a)
ci5_a = (theta5_a - 1.96*se5_a, theta5_a + 1.96*se5_a)

print(f"Estimate:           {theta5_a:.4e}")
print(f"Standard error:     {se5_a:.4e}")
print(f"95% CI:             ({ci5_a[0]:.4e}, {ci5_a[1]:.4e})")
print(f"True value:         {np.exp(-20):.4e}")

Estimate:           0.0000e+00
Standard error:     0.0000e+00
95% CI:             (0.0000e+00, 0.0000e+00)
True value:         2.0612e-09


#### Part b

Importance sampling with $Y \sim \text{Exp}(\lambda)$, $\lambda = 1/10$. The likelihood ratio is

$$\frac{f_X(y)}{f_Y(y)} = \frac{e^{-y}}{\lambda\,e^{-\lambda y}} = \frac{1}{\lambda}\,e^{-y(1-\lambda)}$$

In [24]:
lam5_b = 1/10
U5_b = np.random.uniform(0, 1, size=N5_is)
Y5_b = -np.log(U5_b) / lam5_b  # Exp(lam5_b)
lr5_b = (1/lam5_b) * np.exp(-Y5_b * (1 - lam5_b))
f5_b = (Y5_b > 20) * lr5_b

theta5_b = np.mean(f5_b)
se5_b = np.std(f5_b, ddof=1) / np.sqrt(N5_is)
ci5_b = (theta5_b - 1.96*se5_b, theta5_b + 1.96*se5_b)

print(f"Estimate:           {theta5_b:.4e}")
print(f"Standard error:     {se5_b:.4e}")
print(f"95% CI:             ({ci5_b[0]:.4e}, {ci5_b[1]:.4e})")

Estimate:           2.0773e-09
Standard error:     4.0470e-11
95% CI:             (1.9980e-09, 2.1566e-09)


#### Part c

Repeat with $\lambda = 1/20$ and $\lambda = 1/30$ and compare all three. The proposal mean is $1/\lambda$, so the optimal choice places the bulk of the proposal near the threshold $x = 20$, which corresponds to $\lambda = 1/20$. With $\lambda = 1/10$ the mean is 10, undershooting the threshold and leaving the tail undersampled. With $\lambda = 1/30$ the mean is 30, overshooting so that most draws land well above 20 where the likelihood ratio correction is very small, increasing variance. We therefore expect $\lambda = 1/20$ to give the smallest standard error.

In [25]:
results5 = {}
for lam in [1/10, 1/20, 1/30]:
    U = np.random.uniform(0, 1, size=N5_is)
    Y5 = -np.log(U) / lam  # Exp(lam)
    lr5 = (1/lam) * np.exp(-Y5 * (1 - lam))
    f5 = (Y5 > 20) * lr5
    se = np.std(f5, ddof=1) / np.sqrt(N5_is)
    results5[lam] = (np.mean(f5), se)
    print(f"lam=1/{int(round(1/lam)):2d}: estimate={np.mean(f5):.4e},  SE={se:.4e}")

best = min(results5, key=lambda r: results5[r][1])
print(f"\nSmallest SE: lambda = 1/{int(round(1/best))}")

lam=1/10: estimate=2.0777e-09,  SE=4.0237e-11
lam=1/20: estimate=2.0516e-09,  SE=3.3537e-11
lam=1/30: estimate=2.0400e-09,  SE=3.4429e-11

Smallest SE: lambda = 1/20
